In [1]:
import pandas as pd
import duckdb

df_device_status_log = pd.DataFrame({
    "record_id": [
        901, 902, 903, 904,
        1001, 1002, 1003, 1004,
        1101, 1102, 1103, 1104
    ],
    "device_id": [
        "R05", "R05", "R05", "R05",
        "R16", "R16", "R16", "R16",
        "R34", "R34", "R34", "R34"
    ],
    "record_time": [
        "2026-07-27 08:00:00",
        "2026-07-27 09:00:00",
        "2026-07-27 09:00:00",
        "2026-07-27 10:00:00",

        "2026-07-27 08:30:00",
        "2026-07-27 09:30:00",
        "2026-07-27 10:30:00",
        "2026-07-27 11:30:00",

        "2026-07-27 07:00:00",
        "2026-07-27 08:00:00",
        "2026-07-27 09:00:00",
        "2026-07-27 10:00:00"
    ],
    "status": [
        "OK", "ERROR", "WARNING", "ERROR",
        "ERROR", "ERROR", "OK", "OK",
        "OK", "WARNING", "OK", "ERROR"
    ]
})

df_device_status_log["record_time"] = pd.to_datetime(
    df_device_status_log["record_time"]
)

df_device_status_log

,record_id,device_id,record_time,status
0,901,R05,2026-07-27 08:00:00,OK
1,902,R05,2026-07-27 09:00:00,ERROR
2,903,R05,2026-07-27 09:00:00,WARNING
3,904,R05,2026-07-27 10:00:00,ERROR
4,1001,R16,2026-07-27 08:30:00,ERROR
5,1002,R16,2026-07-27 09:30:00,ERROR
6,1003,R16,2026-07-27 10:30:00,OK
7,1004,R16,2026-07-27 11:30:00,OK
8,1101,R34,2026-07-27 07:00:00,OK
9,1102,R34,2026-07-27 08:00:00,WARNING


# SQL Daily Review：设备累计异常率

## 题目背景

设备会按照时间顺序产生状态记录。

状态包括：

- `OK`：运行正常；
- `WARNING`：存在警告；
- `ERROR`：运行异常。

现在需要观察每台设备从第一条记录到当前记录为止，累计产生了多少条记录、多少条异常记录，以及累计异常率。

## 题目要求

按照每台设备内部的记录顺序，为每条记录计算以下指标：

1. 截至当前记录的累计记录数；
2. 截至当前记录的累计 `ERROR` 记录数；
3. 截至当前记录的累计异常率。

### 记录顺序

每台设备内部按照以下顺序排列：

1. `record_time` 升序；
2. 当 `record_time` 相同时，`record_id` 升序。

### 累计异常率

计算公式为：

```text
累计异常率 = 累计 ERROR 记录数 / 累计记录数
```

结果保留 4 位小数。

只有：

```text
status = 'ERROR'
```

才计入异常记录数。

`WARNING` 不计入异常记录数。

### 输出字段

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号 |
| `record_id` | 记录编号 |
| `record_time` | 记录时间 |
| `status` | 当前状态 |
| `cumulative_record_count` | 截至当前的累计记录数 |
| `cumulative_error_count` | 截至当前的累计异常记录数 |
| `cumulative_error_rate` | 截至当前的累计异常率 |

### 最终排序

按照以下顺序排列：

1. `device_id` 升序；
2. `record_time` 升序；
3. `record_id` 升序。

## 解题要求

- 使用窗口函数完成；
- 使用 `COUNT(*) OVER()` 计算累计记录数；
- 使用条件累计求和计算累计异常记录数；
- 窗口按照 `device_id` 分区；
- 窗口按照 `record_time`、`record_id` 排序；
- 显式指定窗口范围：

```sql
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```

- 使用 CTE 先计算两个累计指标，再在外层计算累计异常率；
- 不使用相关子查询；
- 不使用 `GROUP BY`。

In [ ]:
query = """
WITH status_flag AS (
    SELECT
        device_id,
        record_id,
        record_time,
        status,

        COUNT(*) OVER (
            PARTITION BY device_id
            ORDER BY record_time, record_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_record_count,

        CASE
            WHEN status = 'ERROR' THEN 1
            ELSE 0
        END AS is_error
    FROM df_device_status_log
),

error_count AS (
    SELECT
        device_id,
        record_id,
        record_time,
        status,
        cumulative_record_count,

        SUM(is_error) OVER (
            PARTITION BY device_id
            ORDER BY record_time, record_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )::INTEGER AS cumulative_error_count
    FROM status_flag
)

SELECT
    device_id,
    record_id,
    record_time,
    status,
    cumulative_record_count,
    cumulative_error_count,
    ROUND(
        cumulative_error_count / cumulative_record_count,
        4
    ) AS cumulative_error_rate
FROM error_count
ORDER BY
    device_id,
    record_time,
    record_id;
"""

df = duckdb.execute(query).fetchdf()
df

,device_id,record_id,record_time,status,cumulative_record_count,cumulative_error_count,cumulative_error_rate
0,R34,1101,2026-07-27 07:00:00,OK,1,0,0.0000
1,R34,1102,2026-07-27 08:00:00,WARNING,2,0,0.0000
2,R34,1103,2026-07-27 09:00:00,OK,3,0,0.0000
3,R34,1104,2026-07-27 10:00:00,ERROR,4,1,0.2500
4,R05,901,2026-07-27 08:00:00,OK,1,0,0.0000
5,R05,902,2026-07-27 09:00:00,ERROR,2,1,0.5000
6,R05,903,2026-07-27 09:00:00,WARNING,3,1,0.3333
7,R05,904,2026-07-27 10:00:00,ERROR,4,2,0.5000
8,R16,1001,2026-07-27 08:30:00,ERROR,1,1,1.0000
9,R16,1002,2026-07-27 09:30:00,ERROR,2,2,1.0000
